In [ ]:
import os
import psycopg
import snowflake.connector as sc
import pandas
import json

# Connect to Snowflake

In [ ]:
conn_params = {
    'account': "HOFEOUE-CPB80774",
    'user': "GARRETT@ALIGNABLE.COM",
    'authenticator': "externalbrowser",
    'role': "DEVELOPMENT_ROLE",
    'warehouse': "DEV_SMALL",
    'database': "ALIGNABLE_REPORTING",
    'schema':"EVENT_STREAMS",
    'paramstyle': 'qmark'
}

alignable_reporting_conn = sc.connect(**conn_params)
reporting_cursor = alignable_reporting_conn.cursor()

In [ ]:
def query(query_str):
	return reporting_cursor.execute(query_str).fetch_pandas_all()

In [ ]:
test = query("""
	select * from ALIGNABLE_REPORTING.BUSINESSES.BUSINESSES where id = 8;
""")
test["NAME"]

In [ ]:
from anthropic import AnthropicBedrock
from helpers import OPUS_PROFILE

bedrock_client = AnthropicBedrock(aws_region="us-east-1")

TARGET_TOKENS = 700_000
TOLERANCE = 0.05  # accept within ±5% of target
MAX_ITERATIONS = 5
CHARS_PER_TOKEN = 3.5  # rough Claude average; Bedrock doesn't expose count_tokens

def fetch_messages(row_limit: int):
    return query(f"""
    select *
    from EVENTS.LLM_MESSAGES.LLM_GENERATED_VS_SENT_MESSAGE_EVENTS m
    inner join ALIGNABLE_REPORTING_V2.SPLIT_TESTS.FCT_SPLIT_TEST_ASSIGNMENT a
        on a.business_id = m.source_business_id
    where m.GENERATED_AT >= DATEADD('day', -7, CURRENT_TIMESTAMP())
      and a.split_test_id = 16668
      and a.variation = 'test'
      and m.GENERATED_AT >= a.assigned_at
      and m.SOURCE_BUSINESS_ID != 17589711
    order by RANDOM()
    limit {row_limit};
    """)

def build_context_blob(df) -> str:
    # Wrap each context in a tag with a synthetic, stable ID so the LLM can
    # cite specific examples as evidence. The ID is the row's positional index
    # in the final df — look up via `llm_messages_df.iloc[N]`.
    return "\n\n".join(
        f'<ctx id="CTX-{i:04d}">\n{row}\n</ctx>'
        for i, row in enumerate(df["CONTEXT"].reset_index(drop=True))
    )

def estimate_tokens(text: str) -> int:
    return int(len(text) / CHARS_PER_TOKEN)

limit = 50
llm_messages_df = fetch_messages(limit).reset_index(drop=True)
context_blob = build_context_blob(llm_messages_df)
tokens = estimate_tokens(context_blob)
print(f"limit={limit:>6}  rows={len(llm_messages_df):>6}  ~tokens={tokens:>10,}")

for _ in range(MAX_ITERATIONS):
    if abs(tokens - TARGET_TOKENS) / TARGET_TOKENS <= TOLERANCE:
        break
    tokens_per_row = tokens / max(len(llm_messages_df), 1)
    limit = max(int(TARGET_TOKENS / tokens_per_row), limit + 10)
    llm_messages_df = fetch_messages(limit).reset_index(drop=True)
    context_blob = build_context_blob(llm_messages_df)
    tokens = estimate_tokens(context_blob)
    print(f"limit={limit:>6}  rows={len(llm_messages_df):>6}  ~tokens={tokens:>10,}")

print(f"\nFinal: ~{tokens:,} tokens across {len(llm_messages_df)} rows")
print("(Estimate is char-based; actual count from Bedrock will appear after the Opus call.)")
print("Look up a cited example: llm_messages_df.iloc[<id-int>]  (e.g. CTX-0042 -> .iloc[42])")

In [ ]:
llm_messages_df.head(1)

In [ ]:
# import subprocess

# def copy_to_clipboard(text: str) -> None:
#     subprocess.run("pbcopy", input=text, text=True, check=True)

In [ ]:
# copy_to_clipboard("\n".join(llm_messages_df["CONTEXT"]))

In [ ]:
PROMPT = """\
I work on a professional networking site for small business owners. We have a feature that auto-generates a message from one user (the sender) to another (the target). We want to test inserting a screen *before* generation that explains why the sender might want to message the target — the hope being that a compelling "why" increases the rate at which senders go on to generate and send.

The screen has two parts:
- A short first-sentence hook visible by default.
- A dropdown that, when expanded, reveals 2–4 supporting bullets.

Context about our users: small business owners want more business. On our site that can come from (a) finding customers directly, (b) finding people who can refer them to customers, or (c) finding partners to collaborate with in ways that bring revenue to both sides.

I've attached real context blobs in <context> below. Each one is wrapped in `<ctx id="CTX-####">…</ctx>` and is the context that was used to auto-generate a message between a real sender/target pair on our site. Use them as the source of truth for what reasons actually show up in practice. When you cite a context as evidence, refer to it by its `CTX-####` id (not by quoting the contents).

We will always personalize the copy with the target user's **first name** (derived from `recipient_user_name`). Use the placeholder `{target_first_name}` wherever the first name should appear. Treat that field as always available — do not list it as a "required field" in the audit.

Do this in three steps:

**Step 1 — Pattern analysis.** Read the contexts and identify *every* meaningful reason-pattern that the data actually supports (don't cap the number — find as many as the data justifies; merge near-duplicates). For each pattern, give it a short name, one sentence describing when it applies, and 3–5 example `CTX-####` ids that exemplify it.

**Step 2 — Copy generation.** For each pattern from Step 1, write one piece of copy:
- One first-sentence hook: ≤12 words, second person, warm, concrete, no jargon. Use `{target_first_name}` where natural.
- 2–4 bullets: each ≤10 words, expanding on the hook with specifics a sender would find motivating. Bullets may also use `{target_first_name}`.

**Step 3 — Data-dependency audit.** List every context field the hook + bullets rely on *other than* `{target_first_name}` (e.g., target city, shared industry, target's partner tags, recent activity, etc.). Then, looking across all the contexts in <context>, estimate the share that *lack* at least one required field (e.g., "missing in ~38% — mostly contexts without partner tags"). Be honest about uncertainty; an approximate fraction is fine.

Voice constraints:
- Speak as the platform speaking to a logged-in small business owner.
- Second person ("you"), warm but not saccharine.
- Concrete over abstract. No words like "synergy," "leverage," "ecosystem."
- Don't promise outcomes you can't guarantee ("you will get a customer") — frame as possibility.

Output format: markdown. For each pattern:

```
### Pattern name
One-sentence description.

Example contexts: CTX-####, CTX-####, CTX-####

> *Hook in italics, using {target_first_name} where natural.*
- bullet
- bullet
- bullet

Required fields: …
Missing in dataset: ~N% (brief note on why)
```
"""

response = bedrock_client.messages.create(
    model=OPUS_PROFILE,
    max_tokens=8_000,
    messages=[
        {
            "role": "user",
            "content": f"{PROMPT}\n\n<context>\n{context_blob}\n</context>",
        }
    ],
)

print(f"input tokens:  {response.usage.input_tokens:,}")
print(f"output tokens: {response.usage.output_tokens:,}")
print()
print(response.content[0].text)

In [ ]:
import json

def extract_recipient(ctx_json: str):
    """Pull recipient_business_name and recipient_user_name out of a CONTEXT blob."""
    try:
        data = json.loads(ctx_json)
    except (TypeError, json.JSONDecodeError):
        return {"parse_error": True, "business_name": None, "user_name": None}
    info = (
        data.get("state", {})
            .get("prompt_context", {})
            .get("input", {})
            .get("recipient", {})
            .get("profile_data", {})
            .get("recipient_general_profile_information", {})
    )
    return {
        "parse_error": False,
        "business_name": info.get("recipient_business_name"),
        "user_name": info.get("recipient_user_name"),
    }

recipient_df = llm_messages_df["CONTEXT"].apply(extract_recipient).apply(pandas.Series)
recipient_df["ctx_id"] = [f"CTX-{i:04d}" for i in range(len(recipient_df))]

def classify(row):
    if row["parse_error"]:
        return "parse_error"
    name = row["business_name"]
    if name is None:
        return "missing_key_or_null"
    if isinstance(name, str) and name.strip() == "":
        return "empty_string"
    if isinstance(name, str) and row["user_name"] and name.strip() == row["user_name"].strip():
        return "equals_user_name"
    return "present"

recipient_df["status"] = recipient_df.apply(classify, axis=1)

print("Counts by status:")
print(recipient_df["status"].value_counts())
print()

problem_mask = recipient_df["status"] != "present"
print(f"Rows where recipient_business_name is problematic: {problem_mask.sum()} / {len(recipient_df)} "
      f"({problem_mask.mean():.1%})")
print()
print("Sample of problem rows (ctx_id, status, business_name, user_name):")
recipient_df.loc[problem_mask, ["ctx_id", "status", "business_name", "user_name"]].head(20)

In [ ]:
from collections import Counter

def is_present(value) -> bool:
    """A property is 'present' if it has a meaningful value."""
    if value is None:
        return False
    if isinstance(value, str):
        return value.strip() != ""
    if isinstance(value, (list, dict)):
        return len(value) > 0
    return True  # numbers, bools, etc.

def collect_paths(node, prefix: str = "") -> set[str]:
    """Walk a parsed JSON object and return the set of dotted paths whose values are present.

    Lists are treated as a single path (we don't index into elements) — present if non-empty.
    Nested dicts are walked recursively, contributing both the parent path (if dict non-empty)
    and any present leaf paths within.
    """
    paths: set[str] = set()
    if isinstance(node, dict):
        if prefix and is_present(node):
            paths.add(prefix)
        for key, value in node.items():
            child_prefix = f"{prefix}.{key}" if prefix else key
            if isinstance(value, dict):
                paths |= collect_paths(value, child_prefix)
            elif is_present(value):
                paths.add(child_prefix)
    return paths

path_counter: Counter[str] = Counter()
parse_errors = 0
for ctx in llm_messages_df["CONTEXT"]:
    try:
        path_counter.update(collect_paths(json.loads(ctx)))
    except (TypeError, json.JSONDecodeError):
        parse_errors += 1

total = len(llm_messages_df) - parse_errors
property_frequency_df = (
    pandas.DataFrame(
        [(path, count, count / total) for path, count in path_counter.items()],
        columns=["property", "count", "pct"],
    )
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print(f"Parsed {total}/{len(llm_messages_df)} contexts ({parse_errors} parse errors)")
print(f"Distinct property paths: {len(property_frequency_df)}")
print()
property_frequency_df

In [ ]:
EXISTING_PATTERNS = [
    "Referral Partnership Fit",
    "Target Could Be a Direct Customer",
    "Same Local Area",
    "Mutual Connections",
    "Met at a Networking Event (Online or In-Person)",
    "Introduced by a Mutual Contact",
    "Both Could Send Each Other Business (Two-Way Match)",
    "Same or Related Industry / Role Overlap",
    "Target Serves the Sender's Customers",
    "Shared Community or Group Membership",
    "Known Outside the Platform (Contact List / ACF)",
    "Meeting Already in Progress / Momentum",
]

property_table_md = property_frequency_df.to_markdown(index=False, floatfmt=".3f")

REFINEMENT_PROMPT = f"""\
You already analyzed a set of context blobs from our small-business networking site and produced 12 reason-patterns for why a sender might message a target. The patterns are:

{chr(10).join(f"{i+1}. {p}" for i, p in enumerate(EXISTING_PATTERNS))}

We then ran a separate analysis on the same contexts and measured, for every JSON property path, the share of contexts where that property is present and non-empty. Below is the full frequency table — use these as ground truth instead of estimating:

{property_table_md}

A reviewer looked at which properties your 12 patterns relied on vs. which exist in the data, and flagged the following as **underused, high-frequency signals** that could power new or richer patterns:

- `recipient_customer_section.recipient_customer_tags` (64%) + `recipient_customer_section.recipient_description` (67%) — the target's own description of their ideal customer. Distinct from "target could be your customer": this is "you fit their ICP" or "you serve their customers."
- `recipient_customer_section.recipient_characteristics.recipient_business_model` (86%) + `recipient_geographic_reach` (84%) — useful for "B2B vs B2C match" or "they serve the same region you do" framings.
- `sender_partner_section.sender_partner_tags` (93%) + `sender_partner_section.sender_description` (84%) — the *sender's* stated partner wishlist. Cross-referencing both sides' partner tags (instead of only target's) would tighten Pattern 1.
- `relationship_data.sender_prefers_referral_partners` (50%) + `recipient_prefers_referral_partners` (41%) — explicit booleans. More reliable than inferring intent from partner_tags.
- `relationship_data.same_state` (43%) — useful for "same region" even when `distance_miles` is large.
- `relationship_data.recipient_cares_about_locality` (32%) + `sender_cares_about_locality` (25%) — strengthens "Same Local Area" when *both* sides care.
- `recipient_partner_section.recipient_description` (47%) — target's free-text partner wishlist; concrete claims instead of just "they list your industry."
- `state.connection_request_source` (87%) — the entry point (e.g. `biz/search-more`). Could shape which pattern is most relevant given how the sender arrived.
- `state.conversation_history` (37%) — currently only used for "Meeting in Progress," but also a strong "they already replied to you — follow up" signal in its own right.
- `recipient_how_we_got_started.recipient_description` (29%) + `sender_how_we_got_started.sender_description` (48%) — origin stories; possible "shared origin" pattern (family business, recent founding, etc.).

Now do two things:

**Task A — Additional patterns.** Propose net-new reason-patterns powered primarily by properties not used in the original 12. Don't force it — only propose patterns that the data actually supports. For each, give it a short name, a one-sentence description, 3–5 example `CTX-####` ids, the copy (one hook ≤12 words + 2–4 bullets ≤10 words each, using `{{target_first_name}}` where natural), the required fields, and the exact missing-rate computed from the frequency table above (not estimated).

**Task B — Pattern refinements.** For each of the original 12 patterns where one or more of the underused properties would meaningfully sharpen the trigger or enrich the copy, propose a refinement. For each refinement: name the original pattern, list the additional property/properties to fold in, explain in one sentence what changes (trigger tightening vs. copy enrichment vs. both), and rewrite the hook + bullets accordingly. Recompute the missing-rate using the frequency table. If a pattern doesn't benefit from any underused property, skip it — don't pad.

Voice constraints (same as before):
- Speak as the platform speaking to a logged-in small business owner.
- Second person ("you"), warm but not saccharine.
- Concrete over abstract. No "synergy," "leverage," "ecosystem."
- Don't promise outcomes you can't guarantee.

Output format: markdown.

```
## Task A — Additional patterns

### Pattern name
One-sentence description.

Example contexts: CTX-####, CTX-####, CTX-####

> *Hook in italics, using {{target_first_name}} where natural.*
- bullet
- bullet
- bullet

Required fields: …
Missing in dataset: N% (computed from frequency table — show the worst-covered required field)

## Task B — Pattern refinements

### Original pattern name
Properties added: …
What changes: one sentence.

> *Refined hook in italics.*
- bullet
- bullet
- bullet

Required fields: …
Missing in dataset: N%
```

The contexts are in <context> below, wrapped as before. Cite them by `CTX-####` id.
"""

response2 = bedrock_client.messages.create(
    model=OPUS_PROFILE,
    max_tokens=8_000,
    messages=[
        {
            "role": "user",
            "content": f"{REFINEMENT_PROMPT}\n\n<context>\n{context_blob}\n</context>",
        }
    ],
)

print(f"input tokens:  {response2.usage.input_tokens:,}")
print(f"output tokens: {response2.usage.output_tokens:,}")
print()
print(response2.content[0].text)

In [ ]:
from pathlib import Path

notebook_dir = Path("notebooks/garrett/see_why_wfm")
if not notebook_dir.exists():
    notebook_dir = Path(".")  # fall back to cwd if already inside the notebook dir

output2 = (notebook_dir / "output2.md").read_text()
output3 = (notebook_dir / "output3.md").read_text()

SYNTHESIS_PROMPT = f"""\
You previously produced two passes of analysis on context blobs from our small-business networking site. We now want to merge them into a single, ranked, production-ready list of reason-patterns for why a sender should message a target.

Pass 1 (the original 12 patterns) is in <pass_1>. Pass 2 (additional patterns + refinements) is in <pass_2>. Where Pass 2 refines a Pass 1 pattern, prefer the refined version. Where Pass 2 adds new patterns, include them. Merge near-duplicates.

We also measured the actual presence rate of every JSON property path across the dataset. Use this as ground truth for availability percentages — do not estimate:

{property_table_md}

Produce a single unified list. For each merged pattern, include:

1. **Pattern name** (and a one-sentence description of when it applies).
2. **Properties**, split into:
   - **Necessary**: fields the hook/bullets are nonsensical without. List each with its availability % from the frequency table.
   - **Supplementary**: fields that, if present, allow more specific phrasing — but the copy still works without them. List each with its availability %.
   - **Overall availability**: the share of contexts that have *all* necessary fields. Conservative estimate: use the lowest availability among the necessary fields (or lower, if you know fields are correlated and absences compound).
3. **What's in it for me?** A short paragraph (2–4 sentences) written from the sending business's perspective — answering "why should I, the sender, take time to message this person?" This is the persuasion lever. Be concrete. Avoid jargon ("synergy," "leverage," "ecosystem"). Don't promise outcomes you can't guarantee.
4. **Example copy**:
   - Headline (≤12 words, second person, uses `{{target_first_name}}` where natural).
   - 2–4 bullets (≤10 words each).
5. **Rank rationale**: one sentence on why this pattern lands where it does in the ranking.

Then rank the entire list **by compellingness of "What's in it for me"** from the sender's perspective — strongest motivation first. This is the primary signal for production prioritization. Coverage/availability is *not* the ranking axis, but: if a pattern's overall availability is below ~10%, append a `⚠️ low-coverage` flag after its rank rationale so we can de-prioritize it in production logic regardless of how compelling it is.

Voice constraints (same as before):
- Speak as the platform speaking to a logged-in small business owner.
- Second person ("you"), warm but not saccharine.
- Concrete over abstract.
- Don't promise outcomes you can't guarantee.

Output format: markdown.

```
## Ranked patterns

### 1. Pattern name
*When it applies:* one sentence.

**Properties**
- Necessary: `path.to.field` (XX%), `path.to.other` (XX%)
- Supplementary: `path.to.field` (XX%)
- Overall availability: ~XX%

**What's in it for me?**
A short paragraph from the sender's POV.

**Example**
> *Headline.*
- bullet
- bullet
- bullet

**Rank rationale:** one sentence. [⚠️ low-coverage if overall availability < 10%]

### 2. Pattern name
…
```

<pass_1>
{output2}
</pass_1>

<pass_2>
{output3}
</pass_2>
"""

response3 = bedrock_client.messages.create(
    model=OPUS_PROFILE,
    max_tokens=12_000,
    messages=[
        {"role": "user", "content": SYNTHESIS_PROMPT},
    ],
)

print(f"input tokens:  {response3.usage.input_tokens:,}")
print(f"output tokens: {response3.usage.output_tokens:,}")
print()
print(response3.content[0].text)

In [ ]:
output4 = (notebook_dir / "output4.md").read_text()

FOUR_LENS_PROMPT = f"""\
You previously merged two passes of analysis into a single ranked list of reason-patterns for our message-generation feature. That list is in <ranked_patterns> below. We also have the actual property availability table in <property_frequencies>.

We want each pattern to clearly answer four questions from the sender's perspective:

1. **Why this person?** What about *this specific target* makes them a better recipient than anyone else?
2. **Why this action?** Why is sending a message the right move vs. doing nothing or doing something else?
3. **Why now?** Is there a time-sensitive reason to act today instead of later? (Many patterns are inherently evergreen — if so, mark this lens N/A. Do not manufacture urgency.)
4. **What's in it for me?** What concrete benefit does the sender stand to gain?

Apply this rubric impartially. Don't over-praise the existing copy and don't theatrically tear it apart — judge each pattern against the lens.

For every pattern in <ranked_patterns>, do the following:

**Step 1 — Score.** Rate the *current* copy on each of the four lenses, 1–5, with a one-line justification per lens. Use N/A for "Why now?" on inherently evergreen patterns. A score is about the *copy*, not the underlying signal — a strong signal poorly communicated should score low.

**Step 2 — Rewrite or leave.** If any lens scored ≤3, rewrite the headline + bullets to strengthen the weakest lens(es) — without lengthening the copy or breaking the voice/length constraints. If every lens scored ≥4, leave the copy alone and say so explicitly. Don't rewrite for the sake of rewriting.

**Step 3 — Prune recommendation.** If a pattern is weak on all four lenses AND its overall availability is below ~25%, recommend retiring it. Be concrete about why. Otherwise say "keep."

**Step 4 — Final v2 ranked list.** After processing every pattern, produce a clean v2 ranked list. Re-rank by compellingness of WIIFM (same as the prior pass). Drop any patterns you recommended retiring in Step 3. Keep the same per-pattern structure as the v1 list (when-it-applies, properties, WIIFM, example, rank rationale) — but use the rewritten copy where you rewrote it.

Voice constraints (unchanged):
- Speak as the platform speaking to a logged-in small business owner.
- Second person ("you"), warm but not saccharine.
- Concrete over abstract. No "synergy," "leverage," "ecosystem."
- Don't promise outcomes you can't guarantee.
- Headlines ≤12 words. Bullets ≤10 words each. 2–4 bullets per pattern.

Output format:

```
## Per-pattern review

### Pattern name (current rank N)

**Scores**
- Why this person? X/5 — justification.
- Why this action? X/5 — justification.
- Why now? X/5 or N/A — justification (or "evergreen — no time signal in the data").
- What's in it for me? X/5 — justification.

**Decision:** rewrite | leave-as-is | retire (with one-line reason)

**Rewritten copy (if rewriting):**
> *Headline.*
- bullet
- bullet
- bullet

---

[repeat for every pattern in <ranked_patterns>]

## V2 ranked list

[Re-ranked, pruned, with rewritten copy folded in. Same per-pattern structure as v1: when-it-applies, properties (necessary/supplementary with availability %), overall availability, WIIFM paragraph, example, rank rationale. Apply the ⚠️ low-coverage flag where overall availability < 10%.]
```

<property_frequencies>
{property_table_md}
</property_frequencies>

<ranked_patterns>
{output4}
</ranked_patterns>
"""

response4 = bedrock_client.messages.create(
    model=OPUS_PROFILE,
    max_tokens=16_000,
    messages=[
        {"role": "user", "content": FOUR_LENS_PROMPT},
    ],
)

print(f"input tokens:  {response4.usage.input_tokens:,}")
print(f"output tokens: {response4.usage.output_tokens:,}")
print()
print(response4.content[0].text)

In [ ]:
from pydantic import BaseModel, Field, conlist

class SeeWhyReason(BaseModel):
    rank_and_title: str = Field(
        description='The rank and title of the pattern used, e.g. "3. Target Could Be a Direct Customer". Must match one of the 16 patterns in the prompt exactly.'
    )
    properties_used: list[str] = Field(
        description="Dotted JSON paths (from the context root) of every field used to choose the pattern or write the copy. Example: 'state.prompt_context.input.relationship_data.recipient_matches_sender_customer_targets'."
    )
    headline: str = Field(
        description="The headline shown to the sender. ≤12 words. Uses {target_first_name} where natural, never as a vocative."
    )
    bullets: conlist(str, min_length=2, max_length=2) = Field(
        description="Exactly 2 bullets, each ≤10 words, each grounded in concrete details from this context."
    )

see_why_tool = {
    "name": "emit_see_why_reason",
    "description": "Emit the chosen See Why reason and the copy to show the sender.",
    "input_schema": SeeWhyReason.model_json_schema(),
}

SEE_WHY_PROMPT = (notebook_dir / "SEE_WHY_PROMPT.md").read_text()

sample_context = llm_messages_df["CONTEXT"].iloc[0]

response5 = bedrock_client.messages.create(
    model=OPUS_PROFILE,
    max_tokens=2_000,
    tools=[see_why_tool],
    tool_choice={"type": "tool", "name": "emit_see_why_reason"},
    messages=[
        {
            "role": "user",
            "content": f"{SEE_WHY_PROMPT}\n\n<context>\n{sample_context}\n</context>",
        }
    ],
)

tool_use = next(block for block in response5.content if block.type == "tool_use")
result = SeeWhyReason.model_validate(tool_use.input)

print(f"input tokens:  {response5.usage.input_tokens:,}")
print(f"output tokens: {response5.usage.output_tokens:,}")
print()
print(result.model_dump_json(indent=2))

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from helpers import HAIKU_PROFILE

MAX_CALLS = 500
DRY_RUN = False  # set to False to actually call Bedrock

def generate_see_why(context_str: str) -> SeeWhyReason | None:
    if DRY_RUN:
        return SeeWhyReason(
            rank_and_title="0. DRY RUN (no LLM call)",
            properties_used=["dry.run.placeholder"],
            headline="(dry run) headline placeholder for {target_first_name}",
            bullets=["(dry run) bullet 1", "(dry run) bullet 2"],
        )
    try:
        resp = bedrock_client.messages.create(
            model=HAIKU_PROFILE,
            max_tokens=2_000,
            tools=[see_why_tool],
            tool_choice={"type": "tool", "name": "emit_see_why_reason"},
            messages=[
                {
                    "role": "user",
                    "content": [
                        # Cache the large stable prompt so 2nd-Nth calls only pay for the changing context.
                        {"type": "text", "text": SEE_WHY_PROMPT, "cache_control": {"type": "ephemeral"}},
                        {"type": "text", "text": f"<context>\n{context_str}\n</context>"},
                    ],
                }
            ],
        )
        tool_use = next(b for b in resp.content if b.type == "tool_use")
        return SeeWhyReason.model_validate(tool_use.input)
    except Exception as e:
        print(f"  error: {type(e).__name__}: {e}")
        return None

def format_see_why(r: SeeWhyReason | None) -> str:
    if r is None:
        return ""
    return r.headline + "\n\n" + "\n".join(f"- {b}" for b in r.bullets)

# Cap how many rows we process, regardless of how big llm_messages_df is.
df_to_process = llm_messages_df.head(MAX_CALLS).reset_index(drop=True)
n = len(df_to_process)

print(f"Mode:        {'DRY RUN (no Bedrock calls)' if DRY_RUN else 'LIVE (Haiku)'}")
print(f"Rows in df:  {len(llm_messages_df)}")
print(f"MAX_CALLS:   {MAX_CALLS}")
print(f"Will process {n} rows")
print()

results: list[SeeWhyReason | None] = [None] * n

with ThreadPoolExecutor(max_workers=8) as ex:
    futures = {
        ex.submit(generate_see_why, ctx): i
        for i, ctx in enumerate(df_to_process["CONTEXT"])
    }
    for done, fut in enumerate(as_completed(futures), 1):
        results[futures[fut]] = fut.result()
        print(f"  [{done}/{n}] complete", end="\r")

print()

see_why_csv_df = pandas.DataFrame({
    "generated_message": df_to_process["GENERATED_MESSAGE"].values,
    "context": df_to_process["CONTEXT"].values,
    "see_why_reason": [r.rank_and_title if r else "" for r in results],
    "see_why_formatted": [format_see_why(r) for r in results],
    "properties_used": ["\n".join(r.properties_used) if r else "" for r in results],
})

csv_path = notebook_dir / ("see_why_generations.dry_run.csv" if DRY_RUN else "see_why_generations.csv")
see_why_csv_df.to_csv(csv_path, index=False)
print(f"Wrote {len(see_why_csv_df)} rows to {csv_path}")
print(f"Failures: {sum(1 for r in results if r is None)} / {n}")
see_why_csv_df.head()